In [ ]:
from pathlib import Path
import os

INPUT_ROOT = Path("/kaggle/input")

print("Kaggle input folders:")
for p in INPUT_ROOT.iterdir():
    print("-", p)

print("\nSearching release folder...")
release_folders = list(INPUT_ROOT.rglob("release"))

for p in release_folders:
    print("Found:", p)

if len(release_folders) == 0:
    raise FileNotFoundError("Không tìm thấy thư mục release. Kiểm tra lại file zip/dataset upload lên Kaggle.")

DATA_ROOT = release_folders[0]
print("\nDATA_ROOT =", DATA_ROOT)

print("\nCheck main folders:")
print("list  :", (DATA_ROOT / "list").exists())
print("seg   :", (DATA_ROOT / "seg").exists())
print("origin:", (DATA_ROOT / "origin").exists())

print("\nCheck CSV:")
print("train:", (DATA_ROOT / "list" / "train_fold1.csv").exists())
print("val  :", (DATA_ROOT / "list" / "val_fold1.csv").exists())
print("test :", (DATA_ROOT / "list" / "test.csv").exists())

In [ ]:
import pandas as pd

train_csv = DATA_ROOT / "list" / "train_fold1.csv"
val_csv = DATA_ROOT / "list" / "val_fold1.csv"
test_csv = DATA_ROOT / "list" / "test.csv"

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nFirst rows:")
display(train_df.head())

In [3]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.metrics import roc_auc_score, f1_score


FEATURES = [
    "TonguePale",
    "TipSideRed",
    "Spot",
    "Ecchymosis",
    "Crack",
    "Toothmark",
    "FurThick",
    "FurYellow",
]

TARGETS = [
    "Heart",
    "Lung",
    "Spleen",
    "Liver",
    "Kidney",
]


class TongueDxMultiTaskDataset(Dataset):
    def __init__(self, csv_path, img_root, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_root = Path(img_root)
        self.transform = transform

        required_cols = ["image_path"] + FEATURES + TARGETS
        missing_cols = [c for c in required_cols if c not in self.df.columns]

        if missing_cols:
            raise ValueError(f"CSV thiếu cột: {missing_cols}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.img_root / str(row["image_path"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        feature_label = torch.tensor(
            row[FEATURES].to_numpy(dtype=np.float32),
            dtype=torch.float32
        )

        target_label = torch.tensor(
            row[TARGETS].to_numpy(dtype=np.float32),
            dtype=torch.float32
        )

        return image, feature_label, target_label


def get_transform(img_size=224, train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5),
            transforms.ColorJitter(
                brightness=0.08,
                contrast=0.08,
                saturation=0.08,
                hue=0.02,
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])


class TongueDxMultiTaskModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        try:
            weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            backbone = models.efficientnet_b0(weights=weights)
        except Exception as e:
            print("Không tải được pretrained weights, chuyển sang weights=None.")
            print("Lỗi:", e)
            backbone = models.efficientnet_b0(weights=None)

        in_features = backbone.classifier[1].in_features

        backbone.classifier = nn.Identity()

        self.backbone = backbone

        self.feature_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 8),
        )

        self.target_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 5),
        )

    def forward(self, x):
        emb = self.backbone(x)

        feature_logits = self.feature_head(emb)
        target_logits = self.target_head(emb)

        return feature_logits, target_logits


def make_pos_weight(df, columns):
    pos = df[columns].sum().values.astype(np.float32)
    neg = len(df) - pos
    pos_weight = neg / np.maximum(pos, 1)
    return torch.tensor(pos_weight, dtype=torch.float32)


@torch.no_grad()
def evaluate(model, loader, device, target_thresholds=None, feature_thresholds=None):
    model.eval()

    all_feature_true = []
    all_feature_prob = []

    all_target_true = []
    all_target_prob = []

    for images, feature_labels, target_labels in loader:
        images = images.to(device)

        feature_logits, target_logits = model(images)

        feature_prob = torch.sigmoid(feature_logits).cpu().numpy()
        target_prob = torch.sigmoid(target_logits).cpu().numpy()

        all_feature_true.append(feature_labels.numpy())
        all_feature_prob.append(feature_prob)

        all_target_true.append(target_labels.numpy())
        all_target_prob.append(target_prob)

    feature_true = np.concatenate(all_feature_true, axis=0)
    feature_prob = np.concatenate(all_feature_prob, axis=0)

    target_true = np.concatenate(all_target_true, axis=0)
    target_prob = np.concatenate(all_target_prob, axis=0)

    if feature_thresholds is None:
        feature_thresholds = np.array([0.5] * len(FEATURES))

    if target_thresholds is None:
        target_thresholds = np.array([0.5] * len(TARGETS))

    feature_pred = (feature_prob >= feature_thresholds).astype(int)
    target_pred = (target_prob >= target_thresholds).astype(int)

    feature_auc = {}
    feature_f1 = {}

    target_auc = {}
    target_f1 = {}

    for i, col in enumerate(FEATURES):
        try:
            feature_auc[col] = roc_auc_score(feature_true[:, i], feature_prob[:, i])
        except ValueError:
            feature_auc[col] = np.nan

        feature_f1[col] = f1_score(feature_true[:, i], feature_pred[:, i], zero_division=0)

    for i, col in enumerate(TARGETS):
        try:
            target_auc[col] = roc_auc_score(target_true[:, i], target_prob[:, i])
        except ValueError:
            target_auc[col] = np.nan

        target_f1[col] = f1_score(target_true[:, i], target_pred[:, i], zero_division=0)

    mean_feature_auc = np.nanmean(list(feature_auc.values()))
    mean_target_auc = np.nanmean(list(target_auc.values()))

    mean_feature_f1 = np.mean(list(feature_f1.values()))
    mean_target_f1 = np.mean(list(target_f1.values()))

    return {
        "feature_true": feature_true,
        "feature_prob": feature_prob,
        "target_true": target_true,
        "target_prob": target_prob,
        "feature_auc": feature_auc,
        "feature_f1": feature_f1,
        "target_auc": target_auc,
        "target_f1": target_f1,
        "mean_feature_auc": mean_feature_auc,
        "mean_target_auc": mean_target_auc,
        "mean_feature_f1": mean_feature_f1,
        "mean_target_f1": mean_target_f1,
    }


def find_best_thresholds(y_true, y_prob, n_labels):
    thresholds = []

    for i in range(n_labels):
        best_t = 0.5
        best_f1 = -1

        for t in np.arange(0.05, 0.96, 0.05):
            pred = (y_prob[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], pred, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        thresholds.append(best_t)

    return np.array(thresholds, dtype=np.float32)

In [ ]:
# =====================
# CONFIG
# =====================
IMG_ROOT = DATA_ROOT / "seg"

TRAIN_CSV = DATA_ROOT / "list" / "train_fold1.csv"
VAL_CSV = DATA_ROOT / "list" / "val_fold1.csv"
TEST_CSV = DATA_ROOT / "list" / "test.csv"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 12
LR = 1e-4

FEATURE_LOSS_WEIGHT = 0.4
TARGET_LOSS_WEIGHT = 1.0

OUTPUT_PATH = "/kaggle/working/tonguedx_multitask_best.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_tf = get_transform(IMG_SIZE, train=True)
eval_tf = get_transform(IMG_SIZE, train=False)

train_ds = TongueDxMultiTaskDataset(TRAIN_CSV, IMG_ROOT, train_tf)
val_ds = TongueDxMultiTaskDataset(VAL_CSV, IMG_ROOT, eval_tf)
test_ds = TongueDxMultiTaskDataset(TEST_CSV, IMG_ROOT, eval_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

train_df = pd.read_csv(TRAIN_CSV)

feature_pos_weight = make_pos_weight(train_df, FEATURES).to(device)
target_pos_weight = make_pos_weight(train_df, TARGETS).to(device)

print("Feature pos_weight:")
print(dict(zip(FEATURES, feature_pos_weight.detach().cpu().numpy())))

print("\nTarget pos_weight:")
print(dict(zip(TARGETS, target_pos_weight.detach().cpu().numpy())))

model = TongueDxMultiTaskModel(pretrained=True).to(device)

feature_criterion = nn.BCEWithLogitsLoss(pos_weight=feature_pos_weight)
target_criterion = nn.BCEWithLogitsLoss(pos_weight=target_pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_val_target_auc = -1

best_feature_thresholds = np.array([0.5] * len(FEATURES), dtype=np.float32)
best_target_thresholds = np.array([0.5] * len(TARGETS), dtype=np.float32)


for epoch in range(1, EPOCHS + 1):
    model.train()

    total_loss = 0
    total_feature_loss = 0
    total_target_loss = 0

    for images, feature_labels, target_labels in train_loader:
        images = images.to(device)
        feature_labels = feature_labels.to(device)
        target_labels = target_labels.to(device)

        optimizer.zero_grad()

        feature_logits, target_logits = model(images)

        feature_loss = feature_criterion(feature_logits, feature_labels)
        target_loss = target_criterion(target_logits, target_labels)

        loss = FEATURE_LOSS_WEIGHT * feature_loss + TARGET_LOSS_WEIGHT * target_loss

        loss.backward()
        optimizer.step()

        bs = images.size(0)
        total_loss += loss.item() * bs
        total_feature_loss += feature_loss.item() * bs
        total_target_loss += target_loss.item() * bs

    train_loss = total_loss / len(train_ds)
    train_feature_loss = total_feature_loss / len(train_ds)
    train_target_loss = total_target_loss / len(train_ds)

    val_result = evaluate(model, val_loader, device)

    feature_thresholds = find_best_thresholds(
        val_result["feature_true"],
        val_result["feature_prob"],
        len(FEATURES),
    )

    target_thresholds = find_best_thresholds(
        val_result["target_true"],
        val_result["target_prob"],
        len(TARGETS),
    )

    print("\n" + "=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 80)

    print(f"Train loss        : {train_loss:.4f}")
    print(f"Feature loss      : {train_feature_loss:.4f}")
    print(f"Target loss       : {train_target_loss:.4f}")

    print(f"Val feature AUC   : {val_result['mean_feature_auc']:.4f}")
    print(f"Val target AUC    : {val_result['mean_target_auc']:.4f}")

    print(f"Val feature F1    : {val_result['mean_feature_f1']:.4f}")
    print(f"Val target F1     : {val_result['mean_target_f1']:.4f}")

    print("\nTarget AUC từng nhãn:")
    print(val_result["target_auc"])

    print("\nFeature AUC từng nhãn:")
    print(val_result["feature_auc"])

    print("\nBest target thresholds:")
    print(dict(zip(TARGETS, target_thresholds)))

    if val_result["mean_target_auc"] > best_val_target_auc:
        best_val_target_auc = val_result["mean_target_auc"]
        best_feature_thresholds = feature_thresholds
        best_target_thresholds = target_thresholds

        torch.save({
            "model_state": model.state_dict(),
            "features": FEATURES,
            "targets": TARGETS,
            "img_size": IMG_SIZE,
            "feature_thresholds": best_feature_thresholds,
            "target_thresholds": best_target_thresholds,
            "best_val_target_auc": best_val_target_auc,
        }, OUTPUT_PATH)

        print("\nSaved best model to:", OUTPUT_PATH)

In [ ]:
checkpoint = torch.load(OUTPUT_PATH, map_location=device)

model = TongueDxMultiTaskModel(pretrained=False).to(device)
model.load_state_dict(checkpoint["model_state"])

feature_thresholds = checkpoint["feature_thresholds"]
target_thresholds = checkpoint["target_thresholds"]

test_result = evaluate(
    model,
    test_loader,
    device,
    feature_thresholds=feature_thresholds,
    target_thresholds=target_thresholds,
)

print("=" * 80)
print("TEST RESULT")
print("=" * 80)

print("Mean feature AUC:", test_result["mean_feature_auc"])
print("Mean target AUC :", test_result["mean_target_auc"])

print("\nMean feature F1:", test_result["mean_feature_f1"])
print("Mean target F1 :", test_result["mean_target_f1"])

print("\nTarget AUC từng nhãn:")
for k, v in test_result["target_auc"].items():
    print(f"{k}: {v:.4f}")

print("\nTarget F1 từng nhãn:")
for k, v in test_result["target_f1"].items():
    print(f"{k}: {v:.4f}")

print("\nFeature AUC từng nhãn:")
for k, v in test_result["feature_auc"].items():
    print(f"{k}: {v:.4f}")

print("\nFeature F1 từng nhãn:")
for k, v in test_result["feature_f1"].items():
    print(f"{k}: {v:.4f}")

In [ ]:
OUTPUT_PATH = "/kaggle/working/tonguedx_multitask_best.pt"

checkpoint = torch.load(
    OUTPUT_PATH,
    map_location=device,
    weights_only=False
)

model = TongueDxMultiTaskModel(pretrained=False).to(device)
model.load_state_dict(checkpoint["model_state"])

feature_thresholds = checkpoint["feature_thresholds"]
target_thresholds = checkpoint["target_thresholds"]

test_result = evaluate(
    model,
    test_loader,
    device,
    feature_thresholds=feature_thresholds,
    target_thresholds=target_thresholds,
)

print("=" * 80)
print("TEST RESULT")
print("=" * 80)

print("Mean feature AUC:", test_result["mean_feature_auc"])
print("Mean target AUC :", test_result["mean_target_auc"])

print("\nMean feature F1:", test_result["mean_feature_f1"])
print("Mean target F1 :", test_result["mean_target_f1"])

print("\nTarget AUC từng nhãn:")
for k, v in test_result["target_auc"].items():
    print(f"{k}: {v:.4f}")

print("\nTarget F1 từng nhãn:")
for k, v in test_result["target_f1"].items():
    print(f"{k}: {v:.4f}")

print("\nFeature AUC từng nhãn:")
for k, v in test_result["feature_auc"].items():
    print(f"{k}: {v:.4f}")

print("\nFeature F1 từng nhãn:")
for k, v in test_result["feature_f1"].items():
    print(f"{k}: {v:.4f}")

In [ ]:
from pathlib import Path
from PIL import Image
import torch
import matplotlib.pyplot as plt

VI_FEATURES = {
    "TonguePale": "Lưỡi nhợt",
    "TipSideRed": "Đầu/rìa lưỡi đỏ",
    "Spot": "Có đốm",
    "Ecchymosis": "Ứ huyết / bầm tím",
    "Crack": "Nứt lưỡi",
    "Toothmark": "Dấu răng",
    "FurThick": "Rêu lưỡi dày",
    "FurYellow": "Rêu lưỡi vàng",
}

VI_TARGETS = {
    "Heart": "Tim",
    "Lung": "Phổi",
    "Spleen": "Tỳ / Lá lách",
    "Liver": "Gan",
    "Kidney": "Thận",
}


@torch.no_grad()
def predict_one_image(image_path):
    image_path = Path(image_path)

    image = Image.open(image_path).convert("RGB")

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

    x = eval_tf(image).unsqueeze(0).to(device)

    model.eval()
    feature_logits, target_logits = model(x)

    feature_prob = torch.sigmoid(feature_logits).cpu().numpy()[0]
    target_prob = torch.sigmoid(target_logits).cpu().numpy()[0]

    print("=" * 80)
    print("DỰ ĐOÁN 8 ĐẶC ĐIỂM LƯỠI")
    print("=" * 80)

    for name, prob, th in zip(FEATURES, feature_prob, feature_thresholds):
        status = "Có" if prob >= th else "Không"
        print(f"{VI_FEATURES[name]} ({name}): {status} | prob={prob:.4f} | threshold={th:.2f}")

    print("\n" + "=" * 80)
    print("DỰ ĐOÁN 5 NHÓM CƠ QUAN")
    print("=" * 80)

    for name, prob, th in zip(TARGETS, target_prob, target_thresholds):
        status = "Bất thường" if prob >= th else "Bình thường"
        print(f"{VI_TARGETS[name]} ({name}): {status} | prob={prob:.4f} | threshold={th:.2f}")

    print("\nLưu ý: Kết quả này chỉ phục vụ nghiên cứu/tham khảo, không phải chẩn đoán y khoa.")


test_df = pd.read_csv(TEST_CSV)

sample_idx = 0
sample_path = IMG_ROOT / test_df.iloc[sample_idx]["image_path"]

print("Image path:", sample_path)
predict_one_image(sample_path)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import torch

VI_FEATURES = {
    "TonguePale": "Lưỡi nhợt",
    "TipSideRed": "Đầu/rìa lưỡi đỏ",
    "Spot": "Có đốm",
    "Ecchymosis": "Ứ huyết / bầm tím",
    "Crack": "Nứt lưỡi",
    "Toothmark": "Dấu răng",
    "FurThick": "Rêu lưỡi dày",
    "FurYellow": "Rêu lưỡi vàng",
}

VI_TARGETS = {
    "Heart": "Tim",
    "Lung": "Phổi",
    "Spleen": "Tỳ / Lá lách",
    "Liver": "Gan",
    "Kidney": "Thận",
}


@torch.no_grad()
def predict_image(image_path, title="Ảnh đầu vào"):
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()

    x = eval_tf(image).unsqueeze(0).to(device)

    model.eval()
    feature_logits, target_logits = model(x)

    feature_prob = torch.sigmoid(feature_logits).cpu().numpy()[0]
    target_prob = torch.sigmoid(target_logits).cpu().numpy()[0]

    print("=" * 80)
    print("DỰ ĐOÁN 8 ĐẶC ĐIỂM LƯỠI")
    print("=" * 80)

    for name, prob, th in zip(FEATURES, feature_prob, feature_thresholds):
        status = "Có" if prob >= th else "Không"
        print(f"{VI_FEATURES[name]} ({name}): {status} | prob={prob:.4f} | threshold={th:.2f}")

    print("\n" + "=" * 80)
    print("DỰ ĐOÁN 5 NHÓM CƠ QUAN")
    print("=" * 80)

    for name, prob, th in zip(TARGETS, target_prob, target_thresholds):
        status = "Bất thường" if prob >= th else "Bình thường"
        print(f"{VI_TARGETS[name]} ({name}): {status} | prob={prob:.4f} | threshold={th:.2f}")

    print("\nLưu ý: Kết quả này chỉ phục vụ nghiên cứu/tham khảo, không phải chẩn đoán y khoa.")